# 🐍 Introdução a Python

Neste laboratório vamos trabalhar os principais conceitos da linguagem Python relacionados a Sistemas Distribuídos. 


### Sanity Check do Ambiente

In [ ]:
# Verificação básica do ambiente Python para a disciplina

import sys
import socket
import threading
import asyncio

print("Versão do Python:")
print(sys.version)

print("\nMódulos carregados com sucesso:")
print("socket OK")
print("threading OK")
print("asyncio OK")

### Teste rápido de funcionalidade

In [ ]:
# socket: obter hostname
hostname = socket.gethostname()
print(f"Hostname da máquina: {hostname}")

In [ ]:
# threading: criar uma thread simples
def tarefa():
    print("Thread executando com sucesso")

t = threading.Thread(target=tarefa)
t.start()
t.join()

In [ ]:
# asyncio: loop básico
async def hello():
    print("Asyncio funcionando corretamente")

await hello()

### str vs bytes (demonstração)

In [ ]:
texto = "olá"

print("Tipo original:", type(texto))

In [ ]:
b = texto.encode("utf-8")
print("Convertido para bytes:", b)
print("Tipo:", type(b))

In [ ]:
texto_recuperado = b.decode("utf-8")
print("Decodificado:", texto_recuperado)

### Serialização JSON

In [ ]:
import json

msg = {
    "tipo": "login",
    "payload": {"user": "andré"}
}

json_str = json.dumps(msg)
print("JSON string:", json_str)

In [ ]:
msg_bytes = json_str.encode("utf-8")
print("Bytes:", msg_bytes)

### Cabeçalho de tamanho

In [ ]:
msg = b'{"tipo":"ping"}'

tamanho = len(msg)
print("Tamanho:", tamanho)

In [ ]:
# Representar tamanho com 4 bytes (big-endian)
header = tamanho.to_bytes(4, byteorder="big")

print("Header:", header)

### Função principal: pack_message

In [ ]:
import json

def pack_message(tipo: str, payload: dict) -> bytes:
    """
    Serializa uma mensagem com:
    - JSON
    - Header de 4 bytes com tamanho
    """
    
    # Estrutura da mensagem
    message = {
        "tipo": tipo,
        "payload": payload
    }
    
    # Serialização JSON
    json_str = json.dumps(message)
    
    # Converter para bytes
    msg_bytes = json_str.encode("utf-8")
    
    # Criar header (4 bytes)
    tamanho = len(msg_bytes)
    header = tamanho.to_bytes(4, byteorder="big")
    
    # Retornar mensagem final
    return header + msg_bytes

In [ ]:
msg = pack_message("login", {"user": "ana"})

print("Mensagem final:", msg)

In [ ]:
# Separando header e payload
header = msg[:4]
payload = msg[4:]

print("Header (bytes):", header)
print("Tamanho decodificado:", int.from_bytes(header, "big"))

print("Payload:", payload.decode("utf-8"))

### *args e **kwargs (contexto de rede)

In [ ]:
def handler(tipo, *args, **kwargs):
    print("Tipo:", tipo)
    print("Args:", args)
    print("Kwargs:", kwargs)

handler("login", 123, 456, user="ana", senha="123")

### Comparação prática de estruturas

In [ ]:
import time

dados = list(range(10000000))
dados_dict = {i: i for i in range(10000000)}

# Busca em lista
inicio = time.time()
99999 in dados
print("List:", time.time() - inicio)

In [ ]:
import time

# Busca em dict
inicio = time.time()
99999 in dados_dict
print("Dict:", time.time() - inicio)

### Dataclass para mensagens

In [ ]:
from dataclasses import dataclass

@dataclass
class Message:
    tipo: str
    payload: dict

msg = Message(tipo="login", payload={"user": "ana"})
print(msg)

### Retry com backoff exponencial

In [ ]:
import time
import random

def chamada_remota():
    # Simula falha aleatória
    if random.random() < 0.7:
        raise Exception("Falha de rede")
    return "Sucesso"

def retry(max_tentativas=5):
    for tentativa in range(max_tentativas):
        try:
            resultado = chamada_remota()
            print(resultado)
            return
        except Exception as e:
            wait = 2 ** tentativa
            print(f"Erro: {e} | retry em {wait}s")
            time.sleep(wait)

retry()